<a href="https://colab.research.google.com/github/Hemant10HM/ANNDL-LAB_24mcs004/blob/main/ANN_LAB_10_GNN_Edge_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Showing GNN edge prediction on wikipedia dataset

In [1]:
!pip install torch torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [19]:
import torch
from torch_geometric.datasets import WikipediaNetwork
from torch_geometric.transforms import ToUndirected, RandomLinkSplit
from torch_geometric.nn import GCNConv
import torch.nn.functional as F
from torch.nn import Linear
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

In [20]:
%rm -rf data


In [21]:
# Load dataset
dataset = WikipediaNetwork(root='data/', name='crocodile', geom_gcn_preprocess=False, transform=ToUndirected())

data = dataset[0]

Processing...
Done!


In [22]:
#edge splits using RandomLinkSplit
transform = RandomLinkSplit(is_undirected=True, split_labels=True)
data = transform(data)
train_data, val_data, test_data = data

In [23]:
#GCN encoder class
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [24]:
#edge predictor class
class DotProductDecoder(torch.nn.Module):
    def forward(self, z, edge_index):
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=1)


In [25]:
#Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCNEncoder(dataset.num_node_features, 64).to(device)
decoder = DotProductDecoder().to(device)
train_data, val_data, test_data = train_data.to(device), val_data.to(device), test_data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [26]:
#Training function
def train():
    model.train()
    optimizer.zero_grad()
    z = model(train_data.x, train_data.edge_index)
    pos_score = decoder(z, train_data.pos_edge_label_index)
    neg_edge_index = train_data.neg_edge_label_index
    neg_score = decoder(z, neg_edge_index)

    loss = F.binary_cross_entropy_with_logits(torch.cat([pos_score, neg_score]),
                                              torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)]))
    loss.backward()
    optimizer.step()
    return loss.item()

# Evaluation function
def test(data_split):
    model.eval()
    with torch.no_grad():
        z = model(data_split.x, data_split.edge_index)
        pos_score = decoder(z, data_split.pos_edge_label_index)
        neg_score = decoder(z, data_split.neg_edge_label_index)
        y_true = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).cpu().numpy()
        y_pred = torch.cat([pos_score, neg_score])
        y_prob = torch.sigmoid(y_pred).cpu().numpy()
        y_pred_labels = (y_prob > 0.5).astype(int)

        acc = accuracy_score(y_true, y_pred_labels)
        auc = roc_auc_score(y_true, y_prob)
        precision = precision_score(y_true, y_pred_labels)
        recall = recall_score(y_true, y_pred_labels)

        return acc, auc, precision, recall

In [27]:
# Training loop
for epoch in range(1, 101):
    loss = train()
    val_acc, val_auc, val_prec, val_rec = test(val_data)
    test_acc, test_auc, test_prec, test_rec = test(test_data)
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, AUC: {val_auc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, Test Acc: {test_acc:.4f}, AUC: {test_auc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}")


Epoch: 010, Loss: 0.4732, Val Acc: 0.7144, AUC: 0.9655, Prec: 0.6376, Rec: 0.9935, Test Acc: 0.7173, AUC: 0.9666, Prec: 0.6398, Rec: 0.9944
Epoch: 020, Loss: 0.4186, Val Acc: 0.7737, AUC: 0.9755, Prec: 0.6911, Rec: 0.9898, Test Acc: 0.7775, AUC: 0.9773, Prec: 0.6944, Rec: 0.9912
Epoch: 030, Loss: 0.3973, Val Acc: 0.7854, AUC: 0.9791, Prec: 0.7029, Rec: 0.9887, Test Acc: 0.7838, AUC: 0.9804, Prec: 0.7011, Rec: 0.9895
Epoch: 040, Loss: 0.3888, Val Acc: 0.7944, AUC: 0.9819, Prec: 0.7118, Rec: 0.9894, Test Acc: 0.7943, AUC: 0.9831, Prec: 0.7114, Rec: 0.9904
Epoch: 050, Loss: 0.3838, Val Acc: 0.7966, AUC: 0.9838, Prec: 0.7136, Rec: 0.9908, Test Acc: 0.7982, AUC: 0.9851, Prec: 0.7151, Rec: 0.9916
Epoch: 060, Loss: 0.3804, Val Acc: 0.7996, AUC: 0.9844, Prec: 0.7164, Rec: 0.9917, Test Acc: 0.8004, AUC: 0.9856, Prec: 0.7172, Rec: 0.9921
Epoch: 070, Loss: 0.3777, Val Acc: 0.7999, AUC: 0.9852, Prec: 0.7166, Rec: 0.9923, Test Acc: 0.8013, AUC: 0.9867, Prec: 0.7179, Rec: 0.9927
Epoch: 080, Loss: 0.